# Kandinsky Game→Audio — Step 6: Post-Load Adapter Prototype

Step 5 showed why changing the official loader to unrestricted `strict=False` is unsafe: the factory builds the DiT on the meta device, and an unmatched new parameter may remain uninitialized.

This notebook tests another route on one **real Kandinsky fused block**:
1. Strict-load a synthetic base checkpoint into the unmodified-size block on the meta device.
2. Attach Game→Audio modules only after base loading.
3. Save and reload only those modules as a small sidecar adapter.
4. Confirm base weights stay unchanged and the attached branch can influence audio.

**This is not a pretrained checkpoint, a trained adapter, a pipeline integration, or an audio-generation demo.**

## What a successful run means

It would show that a sidecar adapter can be attached after a strict base load at the fused-block level. It would not yet prove compatibility with the full Kandinsky factory, offload wrappers, real checkpoint files, or a trained event encoder.

## 1. Prepare the pinned source

In [ ]:
import subprocess
from pathlib import Path

import torch
import torch.nn.functional as F
print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
print("Device: CPU-sized dry-run; no GPU is required.")

## 2. Clone and pin Kandinsky source

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-arch-lab")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()
if current != K6_COMMIT:
    subprocess.run(
        ["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT],
        check=True
    )
    subprocess.run(
        ["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT],
        check=True
    )

DIT_PATH = K6_DIR / "kandinsky/core/components/dit.py"
source = DIT_PATH.read_text(encoding="utf-8")
print("Pinned source:", subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip())

## 3. Make a scratch copy with optional Game→Audio support

This uses the already inspected block patch. It changes only the scratch copy in this Colab runtime.

In [ ]:
def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"Expected one source anchor for {label}; found {count}.")
    return text.replace(old, new, 1)

patched = source

patched = replace_once(
    patched,
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n    ):\n",
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n"
    "        game_dim: int | None = None,\n    ):\n",
    "optional game feature dimension",
)

patched = replace_once(
    patched,
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n",
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n"
    "        self.game_dim      = game_dim\n"
    "        if game_dim is None:\n"
    "            self.game_audio_norm = None\n"
    "            self.game_to_audio = None\n"
    "            self.game_audio_modulation = None\n"
    "        else:\n"
    "            self.game_audio_norm = nn.LayerNorm(model_dim_a, elementwise_affine=False)\n"
    "            self.game_to_audio = MultiheadCrossAttention(model_dim_a, head_dim_a, game_dim, engine)\n"
    "            self.game_audio_modulation = Modulation(time_dim_a, model_dim_a, 1)\n",
    "Game→Audio modules",
)

patched = replace_once(
    patched,
    "        av_gate_scale: float = 1.0,\n        va_gate_scale: float = 1.0,\n    ):\n",
    "        av_gate_scale: float = 1.0,\n        va_gate_scale: float = 1.0,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n    ):\n",
    "optional game inputs on FusedTransformerDecoderBlock.forward",
)

patched = replace_once(
    patched,
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # ---- FFN ----\n",
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # Game events condition the audio stream in both audio-only and AV calls.\n"
    "        if game_tokens is not None:\n"
    "            if aud is None:\n"
    "                raise ValueError(\"Game conditioning requires an audio stream.\")\n"
    "            if self.game_to_audio is None:\n"
    "                raise ValueError(\"Construct this block with game_dim to accept game_tokens.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, tokens, game_dim].\")\n"
    "            if game_tokens.shape[0] != aud.shape[0]:\n"
    "                raise ValueError(\"game_tokens and audio must have the same batch size.\")\n"
    "            game_query = self.game_audio_norm(aud).type_as(aud)\n"
    "            game_update = self.game_to_audio(game_query, game_tokens, game_attn_mask)\n"
    "            game_gate = self.game_audio_modulation(t_a)\n"
    "            aud = apply_gate_sum(aud, game_update, game_gate).type_as(aud)\n"
    "\n"
    "        # ---- FFN ----\n",
    "Game→Audio insertion before the audio FFN",
)

PATCH_PATH = DIT_PATH.with_name("dit_game.py")
PATCH_PATH.write_text(patched, encoding="utf-8")
print("Scratch patch written:", PATCH_PATH)
print("Original dit.py unchanged:", DIT_PATH.exists())

## 4. Import Kandinsky's actual fused block with SDPA

In [ ]:
import importlib.util
import sys
import types

if "kandinsky" in sys.modules:
    raise RuntimeError(
        "A Kandinsky package is already imported in this runtime. Restart the runtime, "
        "then run this notebook from the top to keep the source isolation clean."
    )

def register_package(name, path):
    module = types.ModuleType(name)
    module.__path__ = [str(path)]
    module.__package__ = name
    sys.modules[name] = module

register_package("kandinsky", K6_DIR / "kandinsky")
register_package("kandinsky.core", K6_DIR / "kandinsky/core")
register_package("kandinsky.core.components", K6_DIR / "kandinsky/core/components")
register_package("kandinsky.core.components.attention", K6_DIR / "kandinsky/core/components/attention")
register_package("kandinsky.runtime", K6_DIR / "kandinsky/runtime")
register_package("kandinsky.runtime.kernels", K6_DIR / "kandinsky/runtime/kernels")

def pinned_sdpa(q, k, v, attn_mask=None):
    # Same tensor layout and boolean-mask convention as Kandinsky's SDPA wrapper.
    q = q.transpose(1, 2).contiguous()
    k = k.transpose(1, 2).contiguous()
    v = v.transpose(1, 2).contiguous()
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask).transpose(1, 2).contiguous()

class SDPAOnlyEngine:
    def __init__(self, engine="auto"):
        if engine not in {"auto", "sdpa"}:
            raise ValueError("This dry-run intentionally supports only auto/sdpa.")
    def get_attention(self):
        return pinned_sdpa

engine_module = types.ModuleType("kandinsky.runtime.kernels.attention_engine")
engine_module._sdpa = pinned_sdpa
engine_module.SelfAttentionEngine = SDPAOnlyEngine
engine_module.resolve_attention_engine = lambda requested: "sdpa"
sys.modules[engine_module.__name__] = engine_module

def load_dit_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

base_dit = load_dit_module("kandinsky.core.components.dit_base", DIT_PATH)
game_dit = load_dit_module("kandinsky.core.components.dit_game", PATCH_PATH)
print("Loaded block source:", game_dit.__file__)
print("Block class:", game_dit.FusedTransformerDecoderBlock.__name__)

## 5. Create a synthetic base checkpoint

The block dimensions are deliberately small. Its state dict is a stand-in for an old checkpoint: it contains only the original block parameters.

In [ ]:
torch.manual_seed(106)

DIMS = dict(
    model_dim=24,
    time_dim=16,
    ff_dim=48,
    head_dim=6,
    model_dim_a=24,
    time_dim_a=16,
    ff_dim_a=48,
    head_dim_a=6,
    engine="sdpa",
)
GAME_DIM = 7
DEVICE = torch.device("cpu")
DTYPE = torch.float32

base_block = base_dit.FusedTransformerDecoderBlock(**DIMS).eval()
base_checkpoint = {
    name: value.detach().cpu().clone()
    for name, value in base_block.state_dict().items()
}
print("Synthetic base checkpoint keys:", len(base_checkpoint))
print("No downloaded or pretrained weights are used.")

## 6. Strict-load first, then attach the adapter

The patched block is created with `game_dim=None`, so its state dict still has the same keys as the original. We strict-load the base state first. Only after that do we create real Game→Audio modules on CPU.

In [ ]:
with torch.device("meta"):
    loaded_base = game_dit.FusedTransformerDecoderBlock(**DIMS, game_dim=None)

load_result = loaded_base.load_state_dict(
    base_checkpoint,
    strict=True,
    assign=True,
)
assert not load_result.missing_keys
assert not load_result.unexpected_keys

meta_parameters = [
    name for name, parameter in loaded_base.named_parameters()
    if parameter.is_meta
]
assert not meta_parameters, meta_parameters
loaded_base = loaded_base.to(DEVICE).eval()

def attach_game_adapter(block, game_dim, *, device=DEVICE, dtype=DTYPE):
    if block.game_to_audio is not None:
        raise RuntimeError("This block already has a Game→Audio branch.")

    block.game_dim = game_dim
    block.game_audio_norm = torch.nn.LayerNorm(
        block.model_dim_a,
        elementwise_affine=False,
        device=device,
        dtype=dtype,
    )
    block.game_to_audio = game_dit.MultiheadCrossAttention(
        block.model_dim_a,
        DIMS["head_dim_a"],
        game_dim,
        "sdpa",
    ).to(device=device, dtype=dtype)
    block.game_audio_modulation = game_dit.Modulation(
        DIMS["time_dim_a"],
        block.model_dim_a,
        1,
    ).to(device=device, dtype=dtype)
    block.eval()
    return block

attach_game_adapter(loaded_base, GAME_DIM)
assert loaded_base.game_dim == GAME_DIM
assert all(not parameter.is_meta for parameter in loaded_base.parameters())
print("Strict base load: passed")
print("No base parameters left on meta device: passed")
print("Game→Audio branch attached after base load: passed")

## 7. Confirm the base weights did not change

The adapter is new state. Every key from the synthetic base checkpoint should still contain the exact loaded value.

In [ ]:
loaded_state = loaded_base.state_dict()
for name, expected in base_checkpoint.items():
    torch.testing.assert_close(
        loaded_state[name].detach().cpu(),
        expected,
        rtol=0,
        atol=0,
    )
print("All original block weights unchanged after attachment: passed")

new_state_keys = sorted(set(loaded_state) - set(base_checkpoint))
print("New state-dict keys:")
for name in new_state_keys:
    print(" -", name)

## 8. Package only the Game→Audio branch as a sidecar

The sidecar contains the new modules' tensors and no original Kandinsky weights. The gate is opened manually for a serialization demonstration; this is not training.

In [ ]:
import io

def adapter_state_dict(block):
    return {
        "game_audio_norm": {
            key: value.detach().cpu().clone()
            for key, value in block.game_audio_norm.state_dict().items()
        },
        "game_to_audio": {
            key: value.detach().cpu().clone()
            for key, value in block.game_to_audio.state_dict().items()
        },
        "game_audio_modulation": {
            key: value.detach().cpu().clone()
            for key, value in block.game_audio_modulation.state_dict().items()
        },
    }

with torch.no_grad():
    loaded_base.game_audio_modulation.out_layer.bias.fill_(0.1)

sidecar = adapter_state_dict(loaded_base)
buffer = io.BytesIO()
torch.save(sidecar, buffer)
buffer.seek(0)
sidecar_restored = torch.load(
    buffer,
    map_location="cpu",
    weights_only=True,
)
print("Sidecar modules:", list(sidecar_restored))
print("Sidecar contains only Game→Audio modules: passed")

## 9. Load the sidecar into a fresh strict-loaded block

This simulates a later session: first load the same base state strictly, then attach fresh modules, then load only the sidecar into those modules with strict key checks.

In [ ]:
with torch.device("meta"):
    reloaded_base = game_dit.FusedTransformerDecoderBlock(**DIMS, game_dim=None)

reload_result = reloaded_base.load_state_dict(
    base_checkpoint,
    strict=True,
    assign=True,
)
assert not reload_result.missing_keys
assert not reload_result.unexpected_keys
assert all(not parameter.is_meta for parameter in reloaded_base.parameters())
reloaded_base = reloaded_base.to(DEVICE).eval()
attach_game_adapter(reloaded_base, GAME_DIM)

reloaded_base.game_audio_norm.load_state_dict(
    sidecar_restored["game_audio_norm"],
    strict=True,
)
reloaded_base.game_to_audio.load_state_dict(
    sidecar_restored["game_to_audio"],
    strict=True,
)
reloaded_base.game_audio_modulation.load_state_dict(
    sidecar_restored["game_audio_modulation"],
    strict=True,
)

restored_state = adapter_state_dict(reloaded_base)
for module_name in sidecar:
    for key, expected in sidecar[module_name].items():
        torch.testing.assert_close(
            restored_state[module_name][key],
            expected,
            rtol=0,
            atol=0,
        )
print("Strict base + strict sidecar restore: passed")

## 10. Check the attached branch runs

Use the same base checkpoint for a baseline block and an attached block. At the default zero gate, output should match. A manually opened gate should change the audio output.

In [ ]:
batch, audio_len, text_len, game_len = 2, 9, 4, 5
aud = torch.randn(batch, audio_len, DIMS["model_dim_a"])
text_v = torch.randn(batch, text_len, DIMS["model_dim"])
text_a = torch.randn(batch, text_len, DIMS["model_dim_a"])
time_v = torch.randn(batch, DIMS["time_dim"])
time_a = torch.randn(batch, DIMS["time_dim_a"])
time_pair = (time_v, time_a)
game_tokens = torch.randn(batch, game_len, GAME_DIM)
game_valid = torch.tensor([
    [True, True, True, False, False],
    [True, True, False, False, False],
])
game_attn_mask = game_valid[:, None, None, :]
rope_a = base_dit.RoPE1D(DIMS["head_dim_a"])(torch.arange(audio_len))

# First close the gate and confirm the loaded branch is initially a no-op.
with torch.no_grad():
    loaded_base.game_audio_modulation.out_layer.bias.zero_()
    reloaded_base.game_audio_modulation.out_layer.bias.zero_()
    _, baseline_audio = base_block(
        None, aud, text_v, text_a, time_pair, None, rope_a
    )
    _, closed_audio = loaded_base(
        None, aud, text_v, text_a, time_pair, None, rope_a,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    _, restored_closed_audio = reloaded_base(
        None, aud, text_v, text_a, time_pair, None, rope_a,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(closed_audio, baseline_audio, rtol=0, atol=0)
torch.testing.assert_close(restored_closed_audio, baseline_audio, rtol=0, atol=0)
print("Closed adapter gate preserves base output: passed")

# Reopen the demo gate in both blocks; their outputs should match exactly.
with torch.no_grad():
    loaded_base.game_audio_modulation.out_layer.bias.fill_(0.1)
    reloaded_base.game_audio_modulation.out_layer.bias.fill_(0.1)
    _, source_open_audio = loaded_base(
        None, aud, text_v, text_a, time_pair, None, rope_a,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    _, restored_open_audio = reloaded_base(
        None, aud, text_v, text_a, time_pair, None, rope_a,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(restored_open_audio, source_open_audio, rtol=0, atol=0)
audio_delta = (restored_open_audio - baseline_audio).abs().mean().item()
assert audio_delta > 0
print("Mean audio change with demo adapter:", audio_delta)
print("Loaded sidecar reproduces the Game→Audio output: passed")

## What this prototype establishes — and what remains

**Established if all cells pass:** a strict base block load can be followed by post-load branch attachment; only new branch tensors need to be saved in a sidecar; those tensors can be restored independently; the original base weights remain unchanged.

**Still unproven:** full-model attachment across all blocks, Kandinsky factory/offload/compile integration, real checkpoint loading, training, or useful audio control. The manually opened gate is only a plumbing demonstration.

**Next:** extend the attach/load helper across the full tiny DiT stack, then inspect the real pipeline wrapper before modifying its sampler API.